# Improvements.ipynb — RAG Experiment Notebook

Systematically improves the baseline RAG pipeline via 4 sequential experiments.
Each experiment uses the best config selected from the previous one.

| Exp | Component | Search space |
|-----|-----------|--------------|
| A | Embedding model | vietnamese-sbert (baseline) · multilingual-e5-large · vietnamese-bi-encoder |
| B | Chunking strategy | 2000/50 (baseline) · 1000/200 · 500/100 |
| C | Retrieval | top-k ∈ {3,5,7,10} · BM25+vector hybrid (RRF) |
| D | Prompt engineering | simple (baseline) · structured output · few-shot |

**Eval protocol:** same 50 fixed pairs (df.head(50)) for all A/B/C/D experiments  
→ apple-to-apple comparison. 100 pairs only for final evaluation (Step 3).


In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters
!pip install -q faiss-cpu sentence-transformers rank_bm25
!pip install -q pandas numpy tqdm scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 359.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 381.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 310.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 278.1 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
gcsfs 2025.3.0 requires

In [2]:
import os

REPO = "https://github.com/minhbuibhm/rag_chatbot_asm.git"
WORK_DIR = "/kaggle/working/rag_chatbot_asm"

# Clone hoặc pull nếu đã có
if not os.path.exists(WORK_DIR):
    os.system(f"git clone {REPO} {WORK_DIR}")
else:
    os.system(f"cd {WORK_DIR} && git pull")

# Chuyển working directory → relative paths hoạt động đúng
os.chdir(WORK_DIR)
print(f"CWD: {os.getcwd()}")

Cloning into '/kaggle/working/rag_chatbot_asm'...


CWD: /kaggle/working/rag_chatbot_asm


Updating files: 100% (16462/16462), done.


In [3]:
!ls -la /kaggle/input/datasets/minhbhm/minhbhm

ls: cannot access '/kaggle/input/datasets/minhbhm/minhbhm': No such file or directory


In [4]:
import os, re, json, pickle
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_sim

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Constants ──────────────────────────────────────────────────────────
DOCUMENTS_PATH = "Dataset/export_1"
RES_CSV        = "res.csv"
RESULTS_DIR    = "report/results"
N_EVAL         = 50    # fixed for experiments A/B/C/D — do NOT change mid-run
N_FINAL        = 100   # for Step 3 final evaluation only
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# ── Kaggle environment detection ───────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/input/datasets/minhbhm")
if ON_KAGGLE:
    KAGGLE_FAISS   = "/kaggle/input/datasets/minhbhm/llm-rag-asm/faiss_db"
    KAGGLE_CACHE   = "/kaggle/input/datasets/minhbhm/llm-rag-asm/embeddings_cache.pkl"
    KAGGLE_IMPROVEMENTS = "/kaggle/input/datasets/minhbhm/llm-rag-asm-improvements"
    WORK_DIR       = "/kaggle/working"   # writable output directory
    print(f"Kaggle detected — pre-built FAISS: {KAGGLE_FAISS}")
    print(f"Kaggle improvements dataset: {KAGGLE_IMPROVEMENTS}")
else:
    KAGGLE_FAISS   = None
    KAGGLE_CACHE   = None
    KAGGLE_IMPROVEMENTS = None
    WORK_DIR       = "."
    print("Local environment")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

# ── Load evaluation data ───────────────────────────────────────────────
df_full  = pd.read_csv(RES_CSV)
df_eval  = df_full.head(N_EVAL).copy().reset_index(drop=True)
df_final = df_full.head(N_FINAL).copy().reset_index(drop=True)
print(f"Eval set  : {len(df_eval)} pairs  (all experiments)")
print(f"Final set : {len(df_final)} pairs (Step 3 only)")

# ── Baseline prompt ────────────────────────────────────────────────────
def prompt_baseline(question, context):
    return "\n".join([
        "Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh sau.",
        "Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt.",
        "",
        "Ngữ cảnh:",
        context,
        "",
        f"Câu hỏi: {question}",
        "Trả lời ngắn gọn và chính xác nhất có thể:",
    ])

# ── Text normalization ─────────────────────────────────────────────────
def normalize_text(text):
    text = str(text).lower().replace("\n", " ").strip()
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text)

# ── Metric functions ───────────────────────────────────────────────────
def cosine_sim_score(pred, true, emb_model):
    if not pred or not true:
        return 0.0
    return float(sk_cosine_sim([emb_model.embed_query(pred)],
                               [emb_model.embed_query(true)])[0][0])

def jaccard(pred, true):
    p = set(normalize_text(pred).split())
    t = set(normalize_text(true).split())
    return len(p & t) / len(p | t) if p and t else 0.0

def token_overlap(pred, true):
    p = normalize_text(pred).split()
    t = normalize_text(true).split()
    return sum((Counter(p) & Counter(t)).values()) / len(t) if t else 0.0

def bleu(pred, true):
    p = normalize_text(pred).split()
    t = normalize_text(true).split()
    if not p or not t:
        return 0.0
    ov   = sum((Counter(p) & Counter(t)).values())
    prec = ov / len(p)
    bp   = 1.0 if len(p) >= len(t) else np.exp(1 - len(t) / len(p))
    return bp * prec

def rouge_l(pred, true):
    p = normalize_text(pred).split()
    t = normalize_text(true).split()
    if not p or not t:
        return 0.0
    m, n = len(p), len(t)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[i-1] == t[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs  = dp[m][n]
    prec, rec = lcs / m, lcs / n
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def compute_metrics(pred, true, emb_model):
    return {
        "Cosine_Similarity": cosine_sim_score(pred, true, emb_model),
        "Jaccard"          : jaccard(pred, true),
        "Token_Overlap"    : token_overlap(pred, true),
        "BLEU"             : bleu(pred, true),
        "ROUGE_L"          : rouge_l(pred, true),
    }

# ── Core evaluation function ───────────────────────────────────────────
def evaluate_config(vectorstore, emb_model, llm, df, k=5,
                    prompt_fn=prompt_baseline, hybrid_fn=None, label=""):
    # hybrid_fn: optional fn(question, k) -> list[str] replacing vectorstore search
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Eval [{label}]"):
        question     = str(row["Câu Hỏi"]).strip()
        ground_truth = str(row["Trả lời"]).strip()
        try:
            if hybrid_fn is not None:
                context_texts = hybrid_fn(question, k)
            else:
                docs = vectorstore.similarity_search(question, k=k)
                context_texts = [d.page_content for d in docs]
            context  = "\n\n".join(context_texts)
            prompt   = prompt_fn(question, context)
            response = llm.invoke(prompt)
            response = response.strip() if isinstance(response, str) else str(response).strip()
        except Exception as e:
            print(f"  [!] row {idx}: {e}")
            response = ""
        rows.append(compute_metrics(response, ground_truth, emb_model))
    means = pd.DataFrame(rows).mean().to_dict()
    means["label"] = label
    return means

print("Setup complete.")


Kaggle detected — pre-built FAISS: /kaggle/input/datasets/minhbhm/llm-rag-asm/faiss_db
Kaggle improvements dataset: /kaggle/input/datasets/minhbhm/llm-rag-asm-improvements
Device: cuda
Eval set  : 50 pairs  (all experiments)
Final set : 100 pairs (Step 3 only)
Setup complete.


In [5]:
LLM_MODEL_ID = "Qwen/Qwen1.5-1.8B"
print(f"Loading LLM: {LLM_MODEL_ID} ...")

llm = HuggingFacePipeline.from_model_id(
    model_id=LLM_MODEL_ID,
    task="text-generation",
    device_map="auto",
    model_kwargs={"torch_dtype": torch.float16},
    pipeline_kwargs={
        "max_new_tokens" : 512,
        "do_sample"      : False,
        "return_full_text": False,   # return only generated text, NOT the prompt
    },
)
print("LLM ready.")


Loading LLM: Qwen/Qwen1.5-1.8B ...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready.


## Experiment A — Embedding Model

Compare three Vietnamese embedding models. Rebuild FAISS index for each.
Use baseline FAISS (`faiss_db/`) for A1 if it already exists.

> **Note:** `multilingual-e5-large` requires `"query: "` / `"passage: "` prefixes.
> `load_embedding_model()` handles this automatically via subclassing.


In [6]:
def load_embedding_model(model_name):
    """Load HuggingFaceEmbeddings; adds e5 prefixes if needed."""
    is_e5 = "e5" in model_name.lower()
    if is_e5:
        class _E5Embed(HuggingFaceEmbeddings):
            def embed_query(self, text):
                return super().embed_query("query: " + text)
            def embed_documents(self, texts):
                return super().embed_documents(["passage: " + t for t in texts])
        cls = _E5Embed
    else:
        cls = HuggingFaceEmbeddings
    return cls(
        model_name=model_name,
        model_kwargs={"device": DEVICE},
    )


def build_or_load_faiss(emb_model, model_name, chunk_size=2000, chunk_overlap=50):
    """
    Build FAISS or load from cache. Returns vectorstore.
    Search order:
      1. Kaggle baseline dataset:      /kaggle/input/llm-rag-asm/faiss_db  (baseline only)
      2. Kaggle improvements dataset:  /kaggle/input/llm-rag-asm-improvements/faiss_db_<model>_<size>_<overlap>/
      3. Local / WORK_DIR cache
      4. Build from scratch → save to WORK_DIR
    """
    safe      = model_name.split("/")[-1]
    faiss_dir = os.path.join(WORK_DIR, f"faiss_db_{safe}_{chunk_size}_{chunk_overlap}")
    is_baseline = (model_name == "keepitreal/vietnamese-sbert"
                   and chunk_size == 2000 and chunk_overlap == 50)

    # ── Priority 1: Kaggle baseline dataset (legacy faiss_db/) ─────────
    if is_baseline and KAGGLE_FAISS and os.path.exists(KAGGLE_FAISS):
        print(f"Loading Kaggle baseline FAISS ({KAGGLE_FAISS}) ...")
        return FAISS.load_local(KAGGLE_FAISS, emb_model,
                                allow_dangerous_deserialization=True)

    # ── Priority 2: Kaggle improvements dataset ───────────────────────
    if KAGGLE_IMPROVEMENTS:
        kaggle_cfg_dir = os.path.join(KAGGLE_IMPROVEMENTS, f"faiss_db_{safe}_{chunk_size}_{chunk_overlap}")
        if os.path.exists(kaggle_cfg_dir):
            print(f"Loading Kaggle improvements FAISS '{kaggle_cfg_dir}' ...")
            return FAISS.load_local(kaggle_cfg_dir, emb_model,
                                    allow_dangerous_deserialization=True)

    # ── Priority 3: Local / WORK_DIR cache ────────────────────────────
    if is_baseline and os.path.exists("faiss_db"):
        print("Loading local baseline FAISS (faiss_db/) ...")
        return FAISS.load_local("faiss_db", emb_model,
                                allow_dangerous_deserialization=True)

    if os.path.exists(faiss_dir):
        print(f"Loading cached FAISS '{faiss_dir}' ...")
        return FAISS.load_local(faiss_dir, emb_model,
                                allow_dangerous_deserialization=True)

    # ── Priority 4: Build from scratch ─────────────────────────────────
    print(f"Building FAISS | {model_name} | chunk={chunk_size}/{chunk_overlap} ...")
    splitter   = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    all_chunks = []
    files      = sorted(f for f in os.listdir(DOCUMENTS_PATH) if f.endswith(".txt"))
    for fname in tqdm(files, desc="Splitting docs"):
        try:
            content = open(os.path.join(DOCUMENTS_PATH, fname),
                           encoding="utf-8", errors="ignore").read()
            all_chunks.extend(splitter.create_documents([content]))
        except Exception:
            pass
    print(f"  {len(all_chunks)} chunks from {len(files)} docs")

    BATCH = 512
    vs = None
    for i in tqdm(range(0, len(all_chunks), BATCH), desc="Embedding"):
        batch = all_chunks[i:i + BATCH]
        if vs is None:
            vs = FAISS.from_documents(batch, emb_model)
        else:
            vs.add_documents(batch)
    vs.save_local(faiss_dir)
    print(f"  Saved to '{faiss_dir}'")
    return vs

print("Helpers ready.")

Helpers ready.


In [7]:
# A1: keepitreal/vietnamese-sbert (baseline)
MODEL_A1 = "keepitreal/vietnamese-sbert"
emb_A1   = load_embedding_model(MODEL_A1)
vs_A1    = build_or_load_faiss(emb_A1, MODEL_A1)
res_A1   = evaluate_config(vs_A1, emb_A1, llm, df_eval, k=5, label="A1-sbert")
print(res_A1)


/tmp/ipykernel_23/2194576978.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return cls(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading Kaggle baseline FAISS (/kaggle/input/datasets/minhbhm/llm-rag-asm/faiss_db) ...


Eval [A1-sbert]:  20%|██        | 10/50 [03:08<12:28, 18.70s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Eval [A1-sbert]: 100%|██████████| 50/50 [15:19<00:00, 18.38s/it]

{'Cosine_Similarity': 0.5336771253710251, 'Jaccard': 0.12246063298215654, 'Token_Overlap': 0.17563289254492254, 'BLEU': 0.12286598045590537, 'ROUGE_L': 0.12335939006047854, 'label': 'A1-sbert'}


In [8]:
# A2: intfloat/multilingual-e5-large  (e5 prefixes handled by load_embedding_model)
MODEL_A2 = "intfloat/multilingual-e5-large"
emb_A2   = load_embedding_model(MODEL_A2)
vs_A2    = build_or_load_faiss(emb_A2, MODEL_A2)
res_A2   = evaluate_config(vs_A2, emb_A2, llm, df_eval, k=5, label="A2-e5-large")
print(res_A2)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Loading Kaggle improvements FAISS '/kaggle/input/datasets/minhbhm/llm-rag-asm-improvements/faiss_db_multilingual-e5-large_2000_50' ...


Eval [A2-e5-large]: 100%|██████████| 50/50 [13:52<00:00, 16.64s/it]

{'Cosine_Similarity': 0.8978279730003748, 'Jaccard': 0.12894652235708556, 'Token_Overlap': 0.18653884144984573, 'BLEU': 0.1304805085741133, 'ROUGE_L': 0.12275169024007415, 'label': 'A2-e5-large'}


In [9]:
# A3: bkai-foundation-models/vietnamese-bi-encoder
MODEL_A3 = "bkai-foundation-models/vietnamese-bi-encoder"
emb_A3   = load_embedding_model(MODEL_A3)
vs_A3    = build_or_load_faiss(emb_A3, MODEL_A3)
res_A3   = evaluate_config(vs_A3, emb_A3, llm, df_eval, k=5, label="A3-bi-encoder")
print(res_A3)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Loading Kaggle improvements FAISS '/kaggle/input/datasets/minhbhm/llm-rag-asm-improvements/faiss_db_vietnamese-bi-encoder_2000_50' ...


Eval [A3-bi-encoder]: 100%|██████████| 50/50 [15:31<00:00, 18.62s/it]

{'Cosine_Similarity': 0.4040494218775873, 'Jaccard': 0.12130804692394223, 'Token_Overlap': 0.1757369613822785, 'BLEU': 0.12361648028433386, 'ROUGE_L': 0.12011365220736966, 'label': 'A3-bi-encoder'}


In [10]:
# Exp A — Summary
_A_map = {
    "A1-sbert"     : (MODEL_A1, emb_A1, vs_A1),
    "A2-e5-large"  : (MODEL_A2, emb_A2, vs_A2),
    "A3-bi-encoder": (MODEL_A3, emb_A3, vs_A3),
}
df_A = pd.DataFrame([res_A1, res_A2, res_A3]).set_index("label")
print("\n=== Experiment A — Embedding Model Results ===")
print(df_A.round(4).to_string())
df_A.to_csv(f"{RESULTS_DIR}/exp_A_embedding.csv")

# Auto-select by ROUGE_L; override manually if needed
best_A_lbl          = df_A["ROUGE_L"].idxmax()
BEST_EMBEDDING_NAME = _A_map[best_A_lbl][0]
BEST_EMB_MODEL      = _A_map[best_A_lbl][1]
BEST_VS_2000        = _A_map[best_A_lbl][2]   # chunk 2000/50 with best embedding
print(f"\n✓ Best embedding: {best_A_lbl}  ({BEST_EMBEDDING_NAME})")
print("  Override: reassign BEST_EMBEDDING_NAME / BEST_EMB_MODEL / BEST_VS_2000 if needed.")



=== Experiment A — Embedding Model Results ===
               Cosine_Similarity  Jaccard  Token_Overlap    BLEU  ROUGE_L
label                                                                    
A1-sbert                  0.5337   0.1225         0.1756  0.1229   0.1234
A2-e5-large               0.8978   0.1289         0.1865  0.1305   0.1228
A3-bi-encoder             0.4040   0.1213         0.1757  0.1236   0.1201

✓ Best embedding: A1-sbert  (keepitreal/vietnamese-sbert)
  Override: reassign BEST_EMBEDDING_NAME / BEST_EMB_MODEL / BEST_VS_2000 if needed.


## Experiment B — Chunking Strategy

Using best embedding from Exp A. Each config builds a separate FAISS index
(saved to `faiss_db_<model>_<size>_<overlap>/`) to enable re-runs without rebuilding.

> **Note:** B1 reuses the vectorstore already built in Exp A (same model + 2000/50 config).


In [11]:
# B1: chunk 2000/50 (baseline) — reuse BEST_VS_2000 from Exp A
res_B1 = evaluate_config(BEST_VS_2000, BEST_EMB_MODEL, llm, df_eval, k=5, label="B1-2000-50")
print(res_B1)


Eval [B1-2000-50]: 100%|██████████| 50/50 [15:37<00:00, 18.76s/it]

{'Cosine_Similarity': 0.5336771253710251, 'Jaccard': 0.12246063298215654, 'Token_Overlap': 0.17563289254492254, 'BLEU': 0.12286598045590537, 'ROUGE_L': 0.12335939006047854, 'label': 'B1-2000-50'}


In [12]:
# B2: chunk 1000/200
vs_B2  = build_or_load_faiss(BEST_EMB_MODEL, BEST_EMBEDDING_NAME, chunk_size=1000, chunk_overlap=200)
res_B2 = evaluate_config(vs_B2, BEST_EMB_MODEL, llm, df_eval, k=5, label="B2-1000-200")
print(res_B2)


Loading Kaggle improvements FAISS '/kaggle/input/datasets/minhbhm/llm-rag-asm-improvements/faiss_db_vietnamese-sbert_1000_200' ...


Eval [B2-1000-200]: 100%|██████████| 50/50 [14:47<00:00, 17.75s/it]

{'Cosine_Similarity': 0.5748861484244094, 'Jaccard': 0.1342087524241335, 'Token_Overlap': 0.19935492237217783, 'BLEU': 0.14973918927274915, 'ROUGE_L': 0.13382187449762106, 'label': 'B2-1000-200'}


In [13]:
# B3: chunk 500/100
vs_B3  = build_or_load_faiss(BEST_EMB_MODEL, BEST_EMBEDDING_NAME, chunk_size=500, chunk_overlap=100)
res_B3 = evaluate_config(vs_B3, BEST_EMB_MODEL, llm, df_eval, k=5, label="B3-500-100")
print(res_B3)


Building FAISS | keepitreal/vietnamese-sbert | chunk=500/100 ...


Splitting docs: 100%|██████████| 16439/16439 [00:21<00:00, 751.75it/s]


  906796 chunks from 16439 docs


Embedding: 100%|██████████| 1772/1772 [2:29:45<00:00,  5.07s/it]


  Saved to '/kaggle/working/faiss_db_vietnamese-sbert_500_100'


Eval [B3-500-100]: 100%|██████████| 50/50 [13:56<00:00, 16.73s/it]

{'Cosine_Similarity': 0.5758410639023903, 'Jaccard': 0.14021894913845986, 'Token_Overlap': 0.19982475205141956, 'BLEU': 0.14497368641488856, 'ROUGE_L': 0.13346478900268927, 'label': 'B3-500-100'}


In [14]:
# Exp B — Summary
_B_map = {
    "B1-2000-50" : (BEST_VS_2000, 2000, 50),
    "B2-1000-200": (vs_B2,        1000, 200),
    "B3-500-100" : (vs_B3,        500,  100),
}
df_B = pd.DataFrame([res_B1, res_B2, res_B3]).set_index("label")
print("\n=== Experiment B — Chunking Strategy Results ===")
print(df_B.round(4).to_string())
df_B.to_csv(f"{RESULTS_DIR}/exp_B_chunking.csv")

best_B_lbl         = df_B["ROUGE_L"].idxmax()
BEST_VS            = _B_map[best_B_lbl][0]
BEST_CHUNK_SIZE    = _B_map[best_B_lbl][1]
BEST_CHUNK_OVERLAP = _B_map[best_B_lbl][2]
print(f"\n✓ Best chunking: {best_B_lbl}  (size={BEST_CHUNK_SIZE}, overlap={BEST_CHUNK_OVERLAP})")



=== Experiment B — Chunking Strategy Results ===
             Cosine_Similarity  Jaccard  Token_Overlap    BLEU  ROUGE_L
label                                                                  
B1-2000-50              0.5337   0.1225         0.1756  0.1229   0.1234
B2-1000-200             0.5749   0.1342         0.1994  0.1497   0.1338
B3-500-100              0.5758   0.1402         0.1998  0.1450   0.1335

✓ Best chunking: B2-1000-200  (size=1000, overlap=200)


## Experiment C — Retrieval Strategy

**C1:** Top-k tuning — no FAISS rebuild needed, just change `k`.

**C2:** BM25 + vector hybrid via Reciprocal Rank Fusion (RRF).
Corpus is rebuilt from documents using best chunk config (cached as `.pkl`).


In [15]:
# Exp C1 — Top-k tuning
exp_C1_rows = []
for k_val in [3, 5, 7, 10]:
    res = evaluate_config(BEST_VS, BEST_EMB_MODEL, llm, df_eval,
                          k=k_val, label=f"C1-k{k_val}")
    exp_C1_rows.append(res)
    print(f"  k={k_val:2d}  ROUGE_L={res['ROUGE_L']:.4f}  Cosine={res['Cosine_Similarity']:.4f}")

df_C1    = pd.DataFrame(exp_C1_rows).set_index("label")
BEST_K   = int(df_C1["ROUGE_L"].idxmax().split("k")[-1])
print(f"\n✓ Best k = {BEST_K}")


Eval [C1-k3]: 100%|██████████| 50/50 [14:33<00:00, 17.48s/it]


  k= 3  ROUGE_L=0.1378  Cosine=0.5656


Eval [C1-k5]: 100%|██████████| 50/50 [14:56<00:00, 17.92s/it]


  k= 5  ROUGE_L=0.1338  Cosine=0.5749


Eval [C1-k7]: 100%|██████████| 50/50 [15:54<00:00, 19.09s/it]


  k= 7  ROUGE_L=0.1507  Cosine=0.5932


Eval [C1-k10]: 100%|██████████| 50/50 [15:08<00:00, 18.16s/it]

  k=10  ROUGE_L=0.1607  Cosine=0.6336

✓ Best k = 10


In [16]:
from rank_bm25 import BM25Okapi

# Build (or load cached) text corpus for best chunk config
# Save to WORK_DIR so it's writable on Kaggle (/kaggle/working/)
corpus_pkl = os.path.join(WORK_DIR, f"bm25_corpus_{BEST_CHUNK_SIZE}_{BEST_CHUNK_OVERLAP}.pkl")

if os.path.exists(corpus_pkl):
    CORPUS_TEXTS = pickle.load(open(corpus_pkl, "rb"))
    print(f"Loaded BM25 corpus: {len(CORPUS_TEXTS)} chunks")
else:
    print(f"Building corpus (chunk={BEST_CHUNK_SIZE}/{BEST_CHUNK_OVERLAP}) ...")
    splitter     = RecursiveCharacterTextSplitter(
        chunk_size=BEST_CHUNK_SIZE, chunk_overlap=BEST_CHUNK_OVERLAP)
    CORPUS_TEXTS = []
    files        = sorted(f for f in os.listdir(DOCUMENTS_PATH) if f.endswith(".txt"))
    for fname in tqdm(files, desc="Loading docs"):
        try:
            content = open(os.path.join(DOCUMENTS_PATH, fname),
                           encoding="utf-8", errors="ignore").read()
            CORPUS_TEXTS.extend(
                c.page_content for c in splitter.create_documents([content]))
        except Exception:
            pass
    pickle.dump(CORPUS_TEXTS, open(corpus_pkl, "wb"))
    print(f"Corpus: {len(CORPUS_TEXTS)} chunks saved to {corpus_pkl}")

# BM25 index (fast to build, ~30-60s — no caching needed)
print("Building BM25 index ...")
BM25_INDEX = BM25Okapi([t.split() for t in CORPUS_TEXTS])
print("BM25 ready.")


Building corpus (chunk=1000/200) ...


Loading docs: 100%|██████████| 16439/16439 [00:11<00:00, 1478.11it/s]


Corpus: 431873 chunks saved to /kaggle/working/bm25_corpus_1000_200.pkl
Building BM25 index ...
BM25 ready.


In [17]:
# Exp C2 — BM25 + vector hybrid via Reciprocal Rank Fusion (RRF)
def rrf_hybrid_fn(question, k, rrf_k=60):
    # Vector results (over-retrieve, then fuse)
    vec_docs   = BEST_VS.similarity_search(question, k=k * 2)
    vec_ranked = [d.page_content for d in vec_docs]

    # BM25 results
    scores     = BM25_INDEX.get_scores(question.split())
    top_idx    = np.argsort(scores)[::-1][:k * 2]
    bm25_ranked = [CORPUS_TEXTS[i] for i in top_idx]

    # RRF fusion
    rrf_scores = {}
    for rank, text in enumerate(vec_ranked):
        rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (rrf_k + rank + 1)
    for rank, text in enumerate(bm25_ranked):
        rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (rrf_k + rank + 1)
    return sorted(rrf_scores, key=lambda x: -rrf_scores[x])[:k]

res_C2 = evaluate_config(
    BEST_VS, BEST_EMB_MODEL, llm, df_eval,
    k=BEST_K, hybrid_fn=rrf_hybrid_fn, label="C2-bm25-hybrid",
)
print(res_C2)


Eval [C2-bm25-hybrid]: 100%|██████████| 50/50 [20:38<00:00, 24.76s/it]

{'Cosine_Similarity': 0.5829978843814101, 'Jaccard': 0.1437069868209415, 'Token_Overlap': 0.2241218937141863, 'BLEU': 0.1532492186221378, 'ROUGE_L': 0.14036104683768044, 'label': 'C2-bm25-hybrid'}


In [18]:
# Exp C — Summary
all_C_rows = exp_C1_rows + [res_C2]
df_C       = pd.DataFrame(all_C_rows).set_index("label")
print("\n=== Experiment C — Retrieval Strategy Results ===")
print(df_C.round(4).to_string())
df_C.to_csv(f"{RESULTS_DIR}/exp_C_retrieval.csv")

best_C_lbl = df_C["ROUGE_L"].idxmax()
print(f"\n✓ Best retrieval: {best_C_lbl}")

if best_C_lbl == "C2-bm25-hybrid":
    BEST_RETRIEVAL_FN = rrf_hybrid_fn
    print("  → Using BM25 hybrid retrieval")
else:
    BEST_K            = int(best_C_lbl.split('k')[-1])
    BEST_RETRIEVAL_FN = None   # use vectorstore directly
    print(f"  → Using vector search k={BEST_K}")



=== Experiment C — Retrieval Strategy Results ===
                Cosine_Similarity  Jaccard  Token_Overlap    BLEU  ROUGE_L
label                                                                     
C1-k3                      0.5656   0.1427         0.2079  0.1440   0.1378
C1-k5                      0.5749   0.1342         0.1994  0.1497   0.1338
C1-k7                      0.5932   0.1418         0.2293  0.1703   0.1507
C1-k10                     0.6336   0.1616         0.2471  0.1810   0.1607
C2-bm25-hybrid             0.5830   0.1437         0.2241  0.1532   0.1404

✓ Best retrieval: C1-k10
  → Using vector search k=10


## Experiment D — Prompt Engineering

Three prompt variants tested on the best config from A+B+C.
All variants use the same retrieval; only the prompt template changes.

| ID | Variant | Strategy |
|----|---------|----------|
| D1 | Baseline | Simple Vietnamese instruction (same as Baseline.ipynb) |
| D2 | Structured output | Force `[Căn cứ pháp lý]` + `[Nội dung]` format |
| D3 | Few-shot | One in-context Q&A example (pair #51, outside eval set) |

> **Token budget note:** few-shot adds ~200-300 tokens to each prompt.
> With `max_new_tokens=512`, total generation stays within Qwen1.5-1.8B limits.


In [19]:
# Few-shot example: use pair #N_EVAL+1 (outside df_eval to avoid leakage)
fs_row = df_full.iloc[N_EVAL]
FS_Q   = str(fs_row["Câu Hỏi"]).strip()
FS_A   = str(fs_row["Trả lời"]).strip()[:300]   # truncate to stay within token budget

def prompt_structured(question, context):
    return "\n".join([
        "Bạn là trợ lý pháp lý. Trả lời dựa trên văn bản pháp luật dưới đây.",
        "Chỉ dùng thông tin trong ngữ cảnh. Không bịa đặt.",
        "",
        "Ngữ cảnh:",
        context,
        "",
        f"Câu hỏi: {question}",
        "",
        "Trả lời theo format:",
        "[Căn cứ pháp lý]: <tên văn bản/điều khoản liên quan>",
        "[Nội dung]: <câu trả lời chi tiết>",
    ])

def prompt_fewshot(question, context):
    return "\n".join([
        "Bạn là trợ lý pháp lý. Trả lời câu hỏi dựa trên ngữ cảnh được cung cấp.",
        "",
        "Ví dụ:",
        f"Câu hỏi: {FS_Q}",
        f"Trả lời: {FS_A}",
        "",
        "---",
        "Ngữ cảnh:",
        context,
        "",
        f"Câu hỏi: {question}",
        "Trả lời:",
    ])

# Evaluate all prompt variants
_PROMPTS = [
    (prompt_baseline,   "D1-baseline"),
    (prompt_structured, "D2-structured"),
    (prompt_fewshot,    "D3-fewshot"),
]
exp_D_rows = []
for prompt_fn, lbl in _PROMPTS:
    res = evaluate_config(
        BEST_VS, BEST_EMB_MODEL, llm, df_eval,
        k=BEST_K, prompt_fn=prompt_fn,
        hybrid_fn=BEST_RETRIEVAL_FN, label=lbl,
    )
    exp_D_rows.append(res)
    print(f"  {lbl:15s}  ROUGE_L={res['ROUGE_L']:.4f}  Cosine={res['Cosine_Similarity']:.4f}")


Eval [D1-baseline]: 100%|██████████| 50/50 [15:13<00:00, 18.27s/it]


  D1-baseline      ROUGE_L=0.1607  Cosine=0.6336


Eval [D2-structured]: 100%|██████████| 50/50 [03:07<00:00,  3.74s/it]


  D2-structured    ROUGE_L=0.0033  Cosine=0.0560


Eval [D3-fewshot]: 100%|██████████| 50/50 [16:17<00:00, 19.56s/it]

  D3-fewshot       ROUGE_L=0.1520  Cosine=0.6188


In [20]:
# Exp D — Summary
_D_prompt_map = {
    "D1-baseline"  : prompt_baseline,
    "D2-structured": prompt_structured,
    "D3-fewshot"   : prompt_fewshot,
}
df_D = pd.DataFrame(exp_D_rows).set_index("label")
print("\n=== Experiment D — Prompt Engineering Results ===")
print(df_D.round(4).to_string())
df_D.to_csv(f"{RESULTS_DIR}/exp_D_prompt.csv")

best_D_lbl    = df_D["ROUGE_L"].idxmax()
BEST_PROMPT_FN = _D_prompt_map[best_D_lbl]
print(f"\n✓ Best prompt: {best_D_lbl}")



=== Experiment D — Prompt Engineering Results ===
               Cosine_Similarity  Jaccard  Token_Overlap    BLEU  ROUGE_L
label                                                                    
D1-baseline               0.6336   0.1616         0.2471  0.1810   0.1607
D2-structured             0.0560   0.0053         0.0042  0.0028   0.0033
D3-fewshot                0.6188   0.1820         0.2582  0.1901   0.1520

✓ Best prompt: D1-baseline


## Final Comparison & Step 3 — Full Evaluation (100 pairs)

Re-run baseline and best config on 100 pairs for the official report numbers.

**Best config selected:**
- Embedding: `BEST_EMBEDDING_NAME`
- Chunk: `BEST_CHUNK_SIZE` / `BEST_CHUNK_OVERLAP`
- k: `BEST_K`
- Retrieval: `BEST_RETRIEVAL_FN` (None = vector only)
- Prompt: `BEST_PROMPT_FN.__name__`


In [21]:
# Print best config for verification before running the expensive final eval
print("=== Best Configuration ===")
print(f"  Embedding  : {BEST_EMBEDDING_NAME}")
print(f"  Chunk      : {BEST_CHUNK_SIZE} / {BEST_CHUNK_OVERLAP}")
print(f"  Top-k      : {BEST_K}")
print(f"  Retrieval  : {'BM25 hybrid (RRF)' if BEST_RETRIEVAL_FN else 'vector only'}")
print(f"  Prompt     : {BEST_PROMPT_FN.__name__}")

# Override any selection manually before running the next cell if needed:
# BEST_EMBEDDING_NAME = "keepitreal/vietnamese-sbert"
# BEST_EMB_MODEL      = emb_A1
# BEST_VS             = vs_A1
# BEST_CHUNK_SIZE     = 2000
# BEST_CHUNK_OVERLAP  = 50
# BEST_K              = 5
# BEST_RETRIEVAL_FN   = None
# BEST_PROMPT_FN      = prompt_baseline


=== Best Configuration ===
  Embedding  : keepitreal/vietnamese-sbert
  Chunk      : 1000 / 200
  Top-k      : 10
  Retrieval  : vector only
  Prompt     : prompt_baseline


In [22]:
METRICS_COLS = ["Cosine_Similarity", "Jaccard", "Token_Overlap", "BLEU", "ROUGE_L"]

print("Running BASELINE on 100 pairs ...")
res_baseline_100 = evaluate_config(
    vs_A1, emb_A1, llm, df_final,
    k=5, prompt_fn=prompt_baseline,
    hybrid_fn=None, label="BASELINE-100",
)

print("Running BEST CONFIG on 100 pairs ...")
res_best_100 = evaluate_config(
    BEST_VS, BEST_EMB_MODEL, llm, df_final,
    k=BEST_K, prompt_fn=BEST_PROMPT_FN,
    hybrid_fn=BEST_RETRIEVAL_FN, label="BEST-100",
)

# ── Results table ──────────────────────────────────────────────────────
df_final_res = pd.DataFrame([res_baseline_100, res_best_100]).set_index("label")
print("\n=== FINAL RESULTS (100 pairs) ===")
print(df_final_res[METRICS_COLS].round(4).to_string())

print("\n=== IMPROVEMENT DELTA ===")
for col in METRICS_COLS:
    base  = df_final_res.loc["BASELINE-100", col]
    best  = df_final_res.loc["BEST-100", col]
    delta = best - base
    pct   = delta / base * 100 if base > 0 else float('inf')
    print(f"  {col:20s}: {base:.4f} -> {best:.4f}  ({delta:+.4f}, {pct:+.1f}%)")

# ── Save all results ───────────────────────────────────────────────────
df_final_res.to_csv(f"{RESULTS_DIR}/final_evaluation_100pairs.csv")

all_rows = (exp_C1_rows + [res_C2] + exp_D_rows +
            [res_baseline_100, res_best_100])
df_all = pd.DataFrame(
    [res_A1, res_A2, res_A3,
     res_B1, res_B2, res_B3]
    + all_rows
).set_index("label")
df_all.to_csv(f"{RESULTS_DIR}/comparison_table.csv")
print(f"\nAll results saved to {RESULTS_DIR}/")
print("  exp_A_embedding.csv")
print("  exp_B_chunking.csv")
print("  exp_C_retrieval.csv")
print("  exp_D_prompt.csv")
print("  final_evaluation_100pairs.csv")
print("  comparison_table.csv")


Running BASELINE on 100 pairs ...


Eval [BASELINE-100]: 100%|██████████| 100/100 [29:43<00:00, 17.84s/it]


Running BEST CONFIG on 100 pairs ...


Eval [BEST-100]: 100%|██████████| 100/100 [31:18<00:00, 18.78s/it]


=== FINAL RESULTS (100 pairs) ===
              Cosine_Similarity  Jaccard  Token_Overlap    BLEU  ROUGE_L
label                                                                   
BASELINE-100             0.5394   0.1241         0.1892  0.1296   0.1311
BEST-100                 0.5998   0.1621         0.2508  0.1759   0.1585

=== IMPROVEMENT DELTA ===
  Cosine_Similarity   : 0.5394 -> 0.5998  (+0.0604, +11.2%)
  Jaccard             : 0.1241 -> 0.1621  (+0.0380, +30.7%)
  Token_Overlap       : 0.1892 -> 0.2508  (+0.0616, +32.6%)
  BLEU                : 0.1296 -> 0.1759  (+0.0463, +35.7%)
  ROUGE_L             : 0.1311 -> 0.1585  (+0.0275, +21.0%)

All results saved to report/results/
  exp_A_embedding.csv
  exp_B_chunking.csv
  exp_C_retrieval.csv
  exp_D_prompt.csv
  final_evaluation_100pairs.csv
  comparison_table.csv


## Step 4 — Live Demo (5 sample Q&A)

Run the best configuration end-to-end on five Q&A pairs sampled from `res.csv` **outside the 100-pair evaluation window** (rows 101, 151, 201, 301, 501), covering banking charter, lending regulation, payment cards, currency destruction oversight, and fintech policy. Uses `BEST_VS` / `BEST_EMB_MODEL` / `BEST_K=10` / `BEST_PROMPT_FN=prompt_baseline` already defined above.

FAISS is loaded from `/kaggle/input/datasets/minhbhm/llm-rag-asm-improvements/faiss_db_vietnamese-sbert_1000_200/` via `build_or_load_faiss()` (already resolved into `BEST_VS`).

Outputs are printed inline and saved to `report/results/demo_samples.csv` for reference from `report.md` §6.

In [ ]:
# Live demo: 5 sample Q&A through the best config
DEMO_INDICES = [101, 151, 201, 301, 501]
DEMO_LABELS  = [
    "Agribank fundraising (Banking charter)",
    "Draft amendments to Circular 39/2016 (Lending)",
    "VISA card required data (Payment regulation)",
    "Currency destruction oversight council",
    "MoF crypto-asset pilot timeline (Fintech policy)",
]

print(f"Demo config: emb={BEST_EMBEDDING_NAME} | chunk={BEST_CHUNK_SIZE}/{BEST_CHUNK_OVERLAP} | k={BEST_K} | prompt={BEST_PROMPT_FN.__name__}")
print(f"Retrieval   : {'hybrid (' + BEST_RETRIEVAL_FN.__name__ + ')' if BEST_RETRIEVAL_FN else 'pure vector'}
")

demo_rows = []
for idx, topic in zip(DEMO_INDICES, DEMO_LABELS):
    row = df_full.iloc[idx]
    q   = str(row["Câu Hỏi"]).strip()
    gt  = str(row["Trả lời"]).strip()

    # Retrieve top-k chunks (same path as evaluate_config)
    if BEST_RETRIEVAL_FN is not None:
        ctx_texts = BEST_RETRIEVAL_FN(q, BEST_K)
    else:
        docs = BEST_VS.similarity_search(q, k=BEST_K)
        ctx_texts = [d.page_content for d in docs]
    context  = "

".join(ctx_texts)
    prompt   = BEST_PROMPT_FN(q, context)
    response = llm.invoke(prompt)
    response = response.strip() if isinstance(response, str) else str(response).strip()

    metrics  = compute_metrics(response, gt, BEST_EMB_MODEL)

    print("=" * 100)
    print(f"[Row {idx}] {topic}")
    print("=" * 100)
    print(f"Q:  {q}")
    print(f"
Ground truth (first 400 chars):
    {gt[:400]}{'...' if len(gt) > 400 else ''}")
    print(f"
Top-3 retrieved chunk previews:")
    for i, ct in enumerate(ctx_texts[:3], 1):
        print(f"  [{i}] {ct[:180].replace(chr(10), ' ')}...")
    print(f"
Improved-pipeline answer:
    {response[:800]}{'...' if len(response) > 800 else ''}")
    print(f"
Metrics vs ground truth:")
    for mk, mv in metrics.items():
        print(f"  {mk:20s}: {mv:.4f}")
    print()

    demo_rows.append({
        "idx"          : idx,
        "topic"        : topic,
        "question"     : q,
        "ground_truth" : gt,
        "top3_chunks"  : " || ".join(ct[:200].replace("
", " ") for ct in ctx_texts[:3]),
        "answer"       : response,
        **metrics,
    })

df_demo = pd.DataFrame(demo_rows)
demo_csv = f"{RESULTS_DIR}/demo_samples.csv"
df_demo.to_csv(demo_csv, index=False)
print(f"Saved: {demo_csv}")